In [3]:
!pip install chromadb langchain-community langchain-text-splitters langchain-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 k

In [1]:
import os
import json
import requests
import gradio as gr
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
import requests
OPENROUTER_API_KEY = "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"

LLM_MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"

def call_llm(prompt, system_prompt="You are a helpful AI assistant."):
    """Helper function to call the OpenRouter API."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    }
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=data
    )
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error from LLM API: {response.text}"

Cell 3: Knowledge Base & Document Ingestion

In [3]:
global_full_text = ""
chroma_client = chromadb.Client()
collection_name = "assignment_collection"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass
collection = chroma_client.create_collection(name=collection_name)

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def ingest_pdf(file_path):
    global global_full_text

    print(f"Loading {file_path}...")
    loader = PyPDFLoader(file_path)
    documents = loader.load()

    # Save full text for the summarize tool
    global_full_text = "\n".join([doc.page_content for doc in documents])

    print("Chunking document...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    texts = [chunk.page_content for chunk in chunks]

    metadatas = [{"page": str(chunk.metadata.get('page', 'Unknown'))} for chunk in chunks]
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    print("Creating embeddings and storing in ChromaDB...")
    embedded_texts = embeddings_model.embed_documents(texts)
    collection.add(
        embeddings=embedded_texts,
        documents=texts,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Success! Processed {len(documents)} pages into {len(chunks)} searchable chunks.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def summarize_tool(query_ignored):
    print(">> Executing Summarize Tool...")
    truncated_text = global_full_text[:15000]
    prompt = f"Provide a comprehensive, well-structured executive summary of the following document:\n\n{truncated_text}"
    system = "You are an expert summarizer. Abstract the main points clearly."
    return call_llm(prompt, system)

def qa_tool(query):
    print(">> Executing General Q&A Tool...")
    query_embedding = embeddings_model.embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=4
    )

    context = ""
    citations = []

    for i in range(len(results['documents'][0])):
        doc_text = results['documents'][0][i]
        page_num = int(results['metadatas'][0][i]['page']) + 1
        context += f"--- Chunk {i+1} (Page {page_num}) ---\n{doc_text}\n\n"
        citations.append(f"Page {page_num}")

    unique_citations = list(set(citations))

    prompt = f"Answer the user's question based ONLY on the provided context.\n\nContext:\n{context}\n\nQuestion: {query}"
    system = "You are a precise Q&A assistant. If the answer is not in the context, say so."

    answer = call_llm(prompt, system)

    return f"{answer}\n\n**Sources Consulted:** {', '.join(unique_citations)}"

def hindi_translation_tool(query):
    print(">> Executing Hindi Explanation Tool...")
    query_embedding = embeddings_model.embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    context = "\n".join(results['documents'][0])
    prompt = f"Read the following context and answer the user's question. Your final response MUST be entirely in Hindi.\n\nContext:\n{context}\n\nQuestion: {query}"
    system = "You are a bilingual expert. You read English context but explain and answer strictly in Hindi."

    return call_llm(prompt, system)

In [5]:
def agent_dispatcher(query):
    if not global_full_text:
        return "System Error", "Please ingest a PDF document first."

    prompt = f"""Analyze the following user query and classify their intent.
    Return ONLY ONE of the following exact words:
    - SUMMARIZE : If they ask for an overview, summary, or abstract of the whole document.
    - HINDI : If they explicitly ask for an explanation or translation in Hindi.
    - QA : If they are asking a specific question or looking for details in the document.

    Query: "{query}"
    """

    intent_response = call_llm(prompt, system_prompt="You are a strict routing agent. Output only one word.").strip().upper()

    if "SUMMARIZE" in intent_response:
        active_tool = "Summarize Tool"
        response = summarize_tool(query)
    elif "HINDI" in intent_response:
        active_tool = "Hindi Explanation Tool"
        response = hindi_translation_tool(query)
    else:
        active_tool = "General Q&A Tool"
        response = qa_tool(query)

    return active_tool, response

In [6]:
!pip install PyPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 6.6 MB/s eta 0:00:00


In [ ]:
pdf_path = "/content/LLM.pdf"

# Ingest the document
ingest_pdf(pdf_path)

print("\n" + "="*50)

user_query = "Give the summary of this pdf"

# 3. Run the dispatcher
tool_used, final_answer = agent_dispatcher(user_query)

print(f"USER QUERY: {user_query}")
print(f"ROUTED TO TOOL: [{tool_used}]")
print("="*50 + "\n")
print(final_answer)

Loading /content/LLM.pdf...
Chunking document...
Creating embeddings and storing in ChromaDB...
Success! Processed 47 pages into 413 searchable chunks.

>> Executing Summarize Tool...
USER QUERY: Give the summary of this pdf
ROUTED TO TOOL: [Summarize Tool]

**Executive Summary – “A Comprehensive Overview of Large Language Models”**  

---

### 1. Purpose & Scope  
- **Goal:** Offer a concise, self‑contained survey that captures the breadth of recent LLM research (architectural innovations, training strategies, fine‑tuning, multi‑modal extensions, efficiency, evaluation, and challenges).  
- **Target Audience:** Researchers and practitioners seeking a high‑level yet detailed roadmap of the field.  

---

### 2. Key Contributions of the Paper  
| # | Contribution | What It Provides |
|---|--------------|------------------|
| 1 | **Systematic Survey** | A structured narrative of LLM evolution, organized around seven core branches (Pre‑Training, Fine‑Tuning, Efficiency, Inference, Evaluat

In [7]:
pdf_path = "/content/LLM.pdf"

ingest_pdf(pdf_path)

print("\n" + "="*50)


user_query = "What is LLM and What are pretrained models?"


# 3. Run the dispatcher
tool_used, final_answer = agent_dispatcher(user_query)

print(f"USER QUERY: {user_query}")
print(f"ROUTED TO TOOL: [{tool_used}]")
print("="*50 + "\n")
print(final_answer)

Loading /content/LLM.pdf...
Chunking document...
Creating embeddings and storing in ChromaDB...
Success! Processed 47 pages into 413 searchable chunks.

>> Executing General Q&A Tool...
USER QUERY: What is LLM and What are pretrained models?
ROUTED TO TOOL: [General Q&A Tool]

Based on the provided excerpts, the two points are:

**LLM (Large Language Model)**  
- The text refers to “LLMs” throughout the introduction as a class of models that have become a research field on their own.  
- It mentions that LLMs are discussed in terms of “background, pre‑training, fine‑tuning, evaluation, applications, challenges,” etc.  
- In other words, an LLM is a large‑scale language model that serves as a foundation for many downstream research directions (e.g., fine‑tuning, multi‑modal extensions, agents, etc.).

**Pretrained models**  
- The context describes LLMs as being “pre‑trained” on large corpora: “We loosely follow the existing terminology to ensure a standardized outlook of this research 

In [ ]:
pdf_path = "/content/LLM.pdf"

# Ingest the document
ingest_pdf(pdf_path)

print("\n" + "="*50)

user_query = "can you explain me about BERT"

tool_used, final_answer = agent_dispatcher(user_query)

print(f"USER QUERY: {user_query}")
print(f"ROUTED TO TOOL: [{tool_used}]")
print("="*50 + "\n")
print(final_answer)

Loading /content/LLM.pdf...
Chunking document...
Creating embeddings and storing in ChromaDB...
Success! Processed 47 pages into 413 searchable chunks.

>> Executing General Q&A Tool...
USER QUERY: can you explain me about BERT
ROUTED TO TOOL: [General Q&A Tool]

Result: The context provided contains a citation related to BERT ([299] referencing "Roberta: A robustly optimized bert pre-training approach"), but this is an incorrect attribution (RoBERTa is a separate model, not "optimized BERT"). The context does **not** explain what BERT is, its architecture, purpose, or functionality. It only mentions BERT in passing while discussing adversarial fine-tuning and security vulnerabilities in LLMs (as noted in Chunk 2, Page 34: "BERT, adversarial fine-tuning can enhance robustness..."). There is no explanatory content about BERT's definition, development, or technical details in the provided text.

Answer:  
The provided context does not explain what BERT is. It only references BERT in a ci